# Day 2.2 — Keyword Search Baseline
Build the simplest retriever that could possibly work — count the words a question and a chunk
share. A baseline is what later complexity has to beat, and this one fails in a very precise way.

### Step 1 — Turn text into comparable words

`tokenize` lowercases and splits on non-letters, trimming a trailing "s" so *records* matches
*record*. `content_terms` drops stopwords, which carry no evidence.

In [ ]:
STOPWORDS = frozenset("""
    a an and are as at be by can could did do does for from had has have how if in into is it
    its may might must not of on or should that the their them then there these this those to
    was were what when where which who whom why will with you your during been
    """.split())

def tokenize(text):
    """Lowercase words, with a crude plural trim so 'records' matches 'record'."""
    words = re.findall(r"[a-z0-9]+", text.lower())
    return [word[:-1] if word.endswith("s") and len(word) > 4 else word for word in words]

def content_terms(text):
    """The words that carry meaning: no stopwords, nothing shorter than three letters."""
    return {word for word in tokenize(text) if word not in STOPWORDS and len(word) > 2}

question = "At what temperature does charging stop with a warning?"
print("question         :", question)
print("words kept       :", sorted(content_terms(question)))
print("stopwords dropped:", sorted(w for w in question.lower().strip("?").split() if w in STOPWORDS))

### Step 2 — Score every chunk by shared words

The score is the size of the overlap between the question's words and the chunk's. No model, no
vectors, no randomness.

In [ ]:
def keyword_score(question, chunk):
    """How many meaningful words do the question and this chunk share?"""
    return len(content_terms(question) & content_terms(chunk.searchable_text))

def keyword_ranking(question):
    """All 15 chunks, best score first."""
    return sorted(((keyword_score(question, chunk), chunk) for chunk in chunks),
                  key=lambda pair: pair[0], reverse=True)

def show_top(question, k=3):
    print("Q:", question)
    for position, (score, chunk) in enumerate(keyword_ranking(question)[:k], start=1):
        print(f"   rank {position}  score {score:2}  {chunk.chunk_id:42} {chunk.section}")

show_top(question)
best = keyword_ranking(question)[0][1]
print("\nshared words:", sorted(content_terms(question) & content_terms(best.searchable_text)))
print("The question borrowed the document's own vocabulary, so the right section ranks first.")

### Step 3 — Break it with a paraphrase

Ask the same thing in ordinary English. A blackout is what these documents call *islanded
operation*; the loads that keep running are the *priority 1 loads*. Not one word is shared, so this
is a total miss. 2.3 asks the identical question with a different representation.

In [ ]:
PARAPHRASE = "What keeps running during a blackout?"
EXPECTED_CHUNK = "solar_microgrid:load-priorities"          # the section that really answers it

show_top(PARAPHRASE)
ranking = keyword_ranking(PARAPHRASE)
order = [chunk.chunk_id for _, chunk in ranking]
scores = {chunk.chunk_id: score for score, chunk in ranking}

print("\ntop score anywhere in the corpus :", ranking[0][0], "(every chunk scores 0, so the order above")
print("                                    is just the order the chunks were loaded in)")
print("expected chunk                   :", EXPECTED_CHUNK)
print("its keyword score                :", scores[EXPECTED_CHUNK])
print("its rank                         :", order.index(EXPECTED_CHUNK) + 1, "of", len(chunks))
expected = next(chunk for chunk in chunks if chunk.chunk_id == EXPECTED_CHUNK)
print("\nquestion words :", sorted(content_terms(PARAPHRASE)))
print("shared words   :", sorted(content_terms(PARAPHRASE) & content_terms(expected.searchable_text)))
print("the chunk that answers it:", expected.text[:120], "...")

### Try it yourself

Ask the same thing in the documents' own vocabulary and compare the expected chunk's rank.

In [ ]:
# --- Worked solution ---------------------------------------------------------------
REWORDED = "Which loads have priority, and which are shed first?"      # same need, document's words
before = [chunk.chunk_id for _, chunk in keyword_ranking(PARAPHRASE)].index(EXPECTED_CHUNK) + 1
after = [chunk.chunk_id for _, chunk in keyword_ranking(REWORDED)].index(EXPECTED_CHUNK) + 1

show_top(REWORDED)
print(f"\nrank of {EXPECTED_CHUNK}: {before} with the paraphrase -> {after} with the document's words")
print("Keyword search does not need better software here. It needs the user to already know the")
print("document's vocabulary, which is exactly what we cannot assume.")

### Checkpoint

**1. Why did the correct chunk score 0 for "What keeps running during a blackout?"**

<details><summary>Show answer</summary>

Scoring is word overlap, and the chunk contains none of *keeps*, *running* or *blackout*; it says *priority 1 loads*, *emergency lighting*, *shed*. Zero shared words means zero score, however relevant the passage.

</details>

**2. Should we throw keyword search away now?**

<details><summary>Show answer</summary>

No. It is instant, needs no model, and is unbeatable for exact strings: error codes, part numbers, `request_island_mode`. Production systems often run both. We are adding a second signal, not replacing the first.

</details>

### Recap

- **Limitation seen:** a paraphrase gives the correct chunk a score of 0.
- **Layer added:** a lexical baseline whose every score can be recomputed by hand.
- **Evidence:** the same retriever ranks it first once the question reuses the document's vocabulary.